In [29]:
import os
import requests
import pandas as pd
from io import BytesIO
from dotenv import load_dotenv

load_dotenv()


class APIClient:
    """Client pour récupérer les données Vélib et météo."""

    VELIB_URL = (
        "https://opendata.paris.fr/api/explore/v2.1/"
        "catalog/datasets/velib-disponibilite-en-temps-reel/"
        "exports/csv?use_labels=true"
    )

    METEO_URL = (
        "https://public-api.meteofrance.fr/public/DPObs/v2/"
        "station/infrahoraire-6m?id_station=01014002&format=json"
    )

    def __init__(self, dataset: str, timeout: int = 30):
        self.timeout = timeout
        self.session = requests.Session()
        self.choose_dataset(dataset)

    def choose_dataset(self, dataset: str) -> None:
        """Choisit le dataset à récupérer."""
        if dataset == "velib":
            self.dataset_url = self.VELIB_URL
        elif dataset == "meteo":
            self.dataset_url = self.METEO_URL
        else:
            raise ValueError(
                "Dataset inconnu. Choisissez 'velib' ou 'meteo'."
            )

    def get_dataset(self) -> pd.DataFrame:
        """Récupère les données et les convertit en DataFrame."""

        if self.dataset_url == self.VELIB_URL:
            response = self.session.get(
                self.dataset_url,
                timeout=self.timeout,
            )

        else:
            token = os.getenv("TOKEN")

            if not token:
                raise ValueError(
                    "TOKEN absent du fichier .env ou non chargé."
                )

            response = self.session.get(
                self.dataset_url,
                headers={
                    "accept": "*/*",
                    "apikey": token,
                },
                timeout=self.timeout,
            )

        response.raise_for_status()

        if self.dataset_url == self.VELIB_URL:
            return pd.read_csv(
                BytesIO(response.content),
                encoding="utf-8-sig",
                sep=None,
                engine="python",
            )

        data = response.json()
        return pd.json_normalize(data)

    def close(self) -> None:
        self.session.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()

In [ ]:
def get_dataset(dataset: str) -> pd.DataFrame:
    with APIClient(dataset) as client:
        # Récupération des données
        return client.get_dataset()


         lat    lon geo_id_insee        reference_time           insert_time  \
0  46.278167  5.669     01014002  2026-10-16T14:51:05Z  2026-10-16T14:48:55Z   

          validity_time       t      td   u   dd  ...  t_20  t_50  t_100  \
0  2026-10-16T14:48:00Z  296.75  286.85  54  310  ...  None  None   None   

     vv etat_sol sss insolh ray_glo01  pres  pmer  
0  None     None   0   None      None  None  None  

[1 rows x 25 columns]
